# 08 · El modelo de lenguaje como transformador de caracteristicas

> **Objetivo:** mostrar que un LLM puede ser un **transformador mas**
> del pipeline (y nada mas), ejemplificando las buenas practicas
> alrededor: cache, esquema cerrado, inyeccion de dependencias, linea
> base, pruebas sin red.

## Reglas que seguimos (SPEC_llm_transformador.md)

1. Una llamada por valor unico, con cache persistente.
2. Salida validada contra 12 categorias cerradas (Pydantic).
3. Proveedor intercambiable (`FakeLLM` por defecto; cliente real por
   variables de entorno).
4. Evaluacion contra una linea base por palabras clave.
5. Sin fuga: la categoria depende solo del texto del producto.
6. Sin datos personales hacia fuera: solo descripciones (CC BY 4.0).

El OVA no promete que el AUC mejore cuando se añaden las 12 columnas
del LLM al bosque. **Comparamos; reportamos lo que de.**

In [ ]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
if _cwd.name == "legacy":
    PROJECT_ROOT = _cwd.parent.parent
elif _cwd.name == "notebooks":
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:

import polars as pl
from segmentacion.data.cargar import leer_parquet
from segmentacion.data.limpieza import preparar_ventas
from segmentacion.features.llm_clientes import FakeLLM
from segmentacion.features.texto_llm import (
    categorizar_descripciones,
    evaluar,
    regla_palabras_clave,
)

RAW = PROJECT_ROOT / "data" / "raw" / "online_retail_II.parquet"
ventas, canc = preparar_ventas(leer_parquet(RAW), log=False)
print(f"ventas: {ventas.height}  descripciones unicas: {ventas['Description'].n_unique()}")

## 1. Cliente LLM: por defecto, `FakeLLM`

El cliente por defecto en cuadernos y tests es `FakeLLM`, que lee
respuestas canonicas desde un JSON. **Sin red, sin claves.**

Para usar el cliente real (cualquier endpoint compatible con el formato de chat estandar),
descomenta las dos lineas siguientes y configura `.env` siguiendo
`.env.example`. El codigo no nombra proveedores.

In [ ]:

cliente = FakeLLM(PROJECT_ROOT / "tests" / "fixtures" / "respuestas_fake.json")
# cliente = cliente_openai_compatible(
#     base_url=os.environ["LLM_BASE_URL"],
#     api_key_env="LLM_API_KEY",
#     modelo=os.environ["LLM_MODELO"],
# )
print(f"cliente: {type(cliente).__name__}")

## 2. Categorizar las descripciones unicas (con cache)

La cache (`data/interim/categorias_llm.parquet`) es Parquet versionado
y se commitea. Si la cache ya existe, no se vuelve a llamar al LLM.
Para forzar un recalculo, borra la cache.

In [ ]:
CACHE = PROJECT_ROOT / "data" / "interim" / "categorias_llm.parquet"
# Para el cuaderno con FakeLLM, usamos un subconjunto de 3 descripciones
# conocidas (ver tests/fixtures/respuestas_fake.json).
descripciones = pl.Series(
    "Description",
    ["WHITE HANGING HEART T-LIGHT HOLDER", "REGENCY CAKESTAND 3 TIER", "JUMBO BAG RED RETROSPOT"],
)
categorias = categorizar_descripciones(descripciones, cache=CACHE, cliente=cliente, lote=1)
categorias.head()

## 3. Linea base: regla por palabras clave

La regla no usa LLM. Es nuestro **suelo**: si el LLM no supera a la
regla en >=10 puntos de exactitud sobre el gold, no vale la pena.

In [ ]:
regla = regla_palabras_clave(descripciones)
comparacion = pl.DataFrame({
    'Description': descripciones.to_list(),
    'regla': regla['categoria'].to_list(),
    'llm': categorias['categoria'].to_list(),
})
comparacion

## 4. Evaluacion contra el gold set

`tests/fixtures/categorias_gold.csv` tiene 200 descripciones con la
columna `categoria` **vacia**. David la etiqueta a mano. Mientras no se
etiquete, el `evaluar` reporta `n=0`.

In [ ]:
GOLD = PROJECT_ROOT / "tests" / "fixtures" / "categorias_gold.csv"
gold = pl.read_csv(GOLD).filter(pl.col("categoria") != "")
print(f"gold etiquetado: {gold.height} / 200 filas")
if gold.height > 0:
    categorias_todas = regla_palabras_clave(gold["Description"])
    print("regla vs gold:", evaluar(categorias_todas, gold))
    # Comparar con el LLM aqui, cuando la cache incluya las 200 descripciones.
else:
    print("Completa `categorias_gold.csv` a mano antes de evaluar.")

## 5. ¿Mejora el bosque con las 12 columnas del LLM?

**Comparacion que hacemos una vez** (OVA no promete nada).

Para ejecutarla con el dataset completo, hay que:
1. Categorizar las ~5 305 descripciones unicas (con FakeLLM solo cubre
   las del fixture; con cliente real hace ~107 llamadas en lotes de 50).
2. Construir `proporcion_gasto_por_categoria` por cliente y por corte.
3. Fitear dos bosques (con y sin las 12 columnas) y comparar AUC(B).

Lo dejo esbozado abajo; **en entorno de clase se corre con la cache
poblada** (el commit incluye `data/interim/categorias_llm.parquet` tras
una corrida con el cliente real).

In [ ]:
# Esbozo del flujo completo (requiere categorias de las 5305 descripciones):
# cat_full = categorizar_descripciones(
#     ventas['Description'].unique(), cache=CACHE, cliente=cliente, lote=50,
# )
# propA = proporcion_gasto_por_categoria(ventas, cat_full, FECHA_CORTE_A)
# propB = proporcion_gasto_por_categoria(ventas, cat_full, FECHA_CORTE_B)
# A = cohorte(ventas, canc, FECHA_CORTE_A, horizonte=HORIZONTE).join(propA, on='Customer ID', how='left')
# B = cohorte(ventas, canc, FECHA_CORTE_B, horizonte=HORIZONTE).join(propB, on='Customer ID', how='left')
# ... entrenar con `construir_preprocesador(incluir_prop_llm=True)` y comparar.
print("Esbozo. Ver docstring del cuaderno para el flujo completo.")

## Que acabamos de garantizar

- El modulo trata al LLM como **un transformador mas** del pipeline.
- La cache versionada hace reproducible la ejecucion sin red.
- La validacion Pydantic garantiza que **nunca** llega una etiqueta
  inventada al pipeline. Si llega, el modulo marca la fila como
  `invalido` y la categoria queda `otro`.
- La linea base por palabras clave sirve de **suelo**: solo adoptamos
  el LLM si gana por un margen explicito.
- El codigo **no** nombra ningun proveedor: cambiar de uno
  es cambiar tres variables de entorno.

## Preguntas de reflexion

1. Si el proveedor retira el modelo, ¿como re-ejecutar el pipeline?
2. ¿Que pasaria si el LLM devuelve `"accesorio"` en lugar de
   `"accesorios"`? Mira la columna `fuente` tras la ejecucion.
3. ¿Es honesto commitear la cache? ¿Que riesgos tiene no commitearla?